# Solución de Red Neuronal Artificial:
### **AUTOR:** HADSON PAREDES
### **CASO:** Acceso al Crédito y Desempeño Económico de Cooperativas (PRODUCE)

[![License: MIT](https://img.shields.io/badge/License-MIT-yellow.svg)](https://opensource.org/licenses/MIT)
[![Networking: Linkedin](https://img.shields.io/badge/LinkedIn-Hadson%20Paredes-blue?logo=linkedin&style=flat)](https://www.linkedin.com/in/hadson-paredes/) 
[![Networking: Facebook](https://img.shields.io/badge/Facebook-Hadson%20Paredes%20Cordova-Gree?logo=facebook&style=flat)](https://www.facebook.com/hadson.paredescordova/) 
[![Networking: X](https://img.shields.io/badge/Hadson%20Paredes-black?logo=x&style=flat)](https://x.com/hadson_paredes)

## 1. Resumen Descriptivo y Detallado

**Fuente Oficial:** [Plataforma Nacional de Datos Abiertos - PRODUCE](https://www.datosabiertos.gob.pe/dataset/acceso-al-cr%C3%A9dito-y-desempe%C3%B1o-econ%C3%B3mico-de-las-cooperativas-nivel-nacional)

El **Ministerio de la Producción (PRODUCE) del Perú** recopila y publica información estratégica sobre el desempeño económico y el acceso al sistema financiero de las cooperativas a nivel nacional. Este `conjunto de datos` permite analizar la vinculación entre variables empresariales (como ventas promedio, número de trabajadores, sector económico, ubicación geográfica y antigüedad) y el volumen de créditos otorgados a estas entidades.

El **objetivo de esta solución** es _construir, entrenar y evaluar_ una **Red Neuronal Artificial (RNA)** capaz de predecir la probabilidad de que una cooperativa o empresa en el registro alcance un nivel alto de crédito otorgado (`clasificación binaria`), permitiendo a los tomadores de decisiones identificar factores críticos de éxito y optimizar la asignación de recursos financieros.

## 2. Caracterización del Dataset y Preprocesamiento

A continuación, cargamos el dataset `dataset_produce_pe.csv`, analizamos su estructura (`variables y descripción`), valores faltantes, tipos de variables y definimos la estrategia de preprocesamiento y partición sin fuga de información (*data leakage*).

### Variables y descripción:
- `id_coop`: Identificador único de la cooperativa
- `anio`: Año del registro
- `ciiu / descciiu`: Código y descripción de actividad económica (CIIU)
- `sector`: Sector económico (Minería, Servicios, Comercio, etc.)
- `ubigeo / departamento / provincia / distrito`: Ubicación geográfica
- `ventas_prom`: Ventas promedio de la entidad
- `cred_otorgado`: Monto de crédito otorgado (Variable objetivo continua)
- `trabajadores`: Número de trabajadores
- `anio_insc`: Año de inscripción
- `fec_creacion`: Fecha de creación del registro

In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Carga de datos
df = pd.read_csv('dataset_produce_pe.csv', sep='|')
print(f"Dimensiones del dataset (registros, columnas): {df.shape}")
print("Valores faltantes por columna:")
print(df.isnull().sum())

Dimensiones del dataset (registros, columnas): (1034, 14)
Valores faltantes por columna:
id_coop          0
anio             0
ciiu             0
descciiu         0
sector           0
ubigeo           0
departamento     0
provincia        0
distrito         0
ventas_prom      0
cred_otorgado    0
trabajadores     0
anio_insc        0
fec_creacion     0
dtype: int64


In [3]:
# Visualizar los 10 primeros registros del dataset
top_ten_df = df.head(10)
display(top_ten_df)

,id_coop,anio,ciiu,descciiu,sector,ubigeo,departamento,provincia,distrito,ventas_prom,cred_otorgado,trabajadores,anio_insc,fec_creacion,cred_alto
0,07F866F072872A3DA665309932A6D4B9,2021,1429,EXPLOTACION DE OTRAS MINAS Y CANTERAS NCP,MINERIA,211002,PUNO,SAN ANTONIO DE PUTINA,ANANEA,2970000,1.089500e+02,53,2017,20260210,0
1,25549658E8C46F8ABE7987E1571DA568,2019,6023,TRANSPORTE DE CARGA POR CARRETERA,SERVICIOS,211002,PUNO,SAN ANTONIO DE PUTINA,ANANEA,8400000,1.518400e+02,61,1993,20260210,0
2,2E8D1A29824BFA7C9151AA6E26DD512E,2022,1320,EXTRACCION DE MINERALES METALIFEROS NO FERROSO...,MINERIA,211002,PUNO,SAN ANTONIO DE PUTINA,ANANEA,11960000,7.057850e+05,55,2012,20260210,1
3,2E8D1A29824BFA7C9151AA6E26DD512E,2020,1429,EXPLOTACION DE OTRAS MINAS Y CANTERAS NCP,MINERIA,211002,PUNO,SAN ANTONIO DE PUTINA,ANANEA,8600000,2.412534e+06,56,2012,20260210,1
4,2E8D1A29824BFA7C9151AA6E26DD512E,2021,1320,EXTRACCION DE MINERALES METALIFEROS NO FERROSO...,MINERIA,211002,PUNO,SAN ANTONIO DE PUTINA,ANANEA,4400,1.927452e+06,56,2012,20260210,1
5,3E876D66730845BD02E550EFAE6ACB12,2023,1320,EXTRACCION DE MINERALES METALIFEROS NO FERROSO...,MINERIA,211002,PUNO,SAN ANTONIO DE PUTINA,ANANEA,9900000,1.550000e+00,52,2010,20260210,0
6,428BA378F31DE7026BE48B24D37BBBEA,2024,6023,TRANSPORTE DE CARGA POR CARRETERA,SERVICIOS,211002,PUNO,SAN ANTONIO DE PUTINA,ANANEA,13390000,6.206251e+05,58,1994,20260210,1
7,D4758868992E6D60E42B8C3E85D0D818,2020,1320,EXTRACCION DE MINERALES METALIFEROS NO FERROSO...,MINERIA,211002,PUNO,SAN ANTONIO DE PUTINA,ANANEA,5482500,6.033050e+04,54,2002,20260210,0
8,D4758868992E6D60E42B8C3E85D0D818,2019,1320,EXTRACCION DE MINERALES METALIFEROS NO FERROSO...,MINERIA,211002,PUNO,SAN ANTONIO DE PUTINA,ANANEA,5355000,2.580190e+05,38,2002,20260210,1
9,D86D59908AB714DFD72FCEE08768F13D,2019,6519,OTROS TIPOS DE INTERMEDIACION MONETARIA,SERVICIOS,30201,APURIMAC,ANDAHUAYLAS,ANDAHUAYLAS,16800,4.551000e+01,130,1999,20260210,0


### Creación de Variable Objetivo y Balance de Clases

Para transformar este problema en una tarea de clasificación supervisada, definiremos una variable binaria **`cred_alto`**:
- `1`: Si el crédito otorgado (`cred_otorgado`) es mayor o igual a la mediana del conjunto.
- `0`: Si el crédito otorgado es menor a la mediana.

In [2]:
# Definición de variable objetivo binaria basada en la mediana
mediana_credito = df['cred_otorgado'].median()
df['cred_alto'] = (df['cred_otorgado'] >= mediana_credito).astype(int)

print(f"Mediana del crédito otorgado: {mediana_credito}")
print("Balance de clases (cred_alto):")
print(df['cred_alto'].value_counts(normalize=True))

Mediana del crédito otorgado: 106729.95
Balance de clases (cred_alto):
cred_alto
0    0.5
1    0.5
Name: proportion, dtype: float64


### Selección de Características y División Train/Val sin Fuga de Información

Seleccionamos las variables predictoras numéricas y categóricas clave (`ventas_prom`, `trabajadores`, `anio`, `anio_insc`, codificación one-hot para `sector`). El escalado de características (`StandardScaler`) se ajustará **exclusivamente** sobre el conjunto de entrenamiento (`X_train`) para evitar cualquier fuga de información hacia el conjunto de validación.

In [4]:
# Codificación de variables categóricas (Sector)
df_model = pd.get_dummies(df[['ventas_prom', 'trabajadores', 'anio', 'anio_insc', 'sector', 'cred_alto']], columns=['sector'], drop_first=True)

X = df_model.drop(columns=['cred_alto']).values.astype(np.float32)
y = df_model['cred_alto'].values.astype(np.float32).reshape(-1, 1)

# Partición 80% entrenamiento, 20% validación
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Escalado sin fuga de información
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

print(f"Dimensiones X_train: {X_train.shape}, X_val: {X_val.shape}")

Dimensiones X_train: (827, 10), X_val: (207, 10)


## 3. Implementación de la Red Neuronal en PyTorch

Diseñamos una arquitectura de red neuronal feedforward con dos capas ocultas. 

### Justificación de Componentes:
- **Capas Ocultas y Función de Activación ReLU (`nn.ReLU()`):** Se emplea ReLU en las capas ocultas por su capacidad para mitigar el problema del desvanecimiento del gradiente (*vanishing gradient*), permitiendo un aprendizaje rápido y eficiente en redes multicapa.
- **Capa de Salida y Función de Activación Sigmoid (`nn.Sigmoid()`):** Al tratarse de un problema de clasificación binaria (`cred_alto`), la capa de salida consta de una sola neurona con activación sigmoide para acotar la salida al intervalo $[0, 1]$, interpretándose como la probabilidad estimada.
- **Función de Costo (Loss Function):** Se utiliza **`BCELoss`** (Binary Cross-Entropy Loss), idónea para medir la divergencia entre las probabilidades predichas y las etiquetas binarias reales.

In [5]:
class CreditNet(nn.Module):
    def __init__(self, input_dim):
        super(CreditNet, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.sigmoid(out)
        return out

input_dim = X_train.shape[1]
model = CreditNet(input_dim)
print(model)

CreditNet(
  (fc1): Linear(in_features=10, out_features=32, bias=True)
  (relu1): ReLU()
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (relu2): ReLU()
  (fc3): Linear(in_features=16, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)


## 4. Entrenamiento del Modelo

Entrenamos el modelo utilizando el optimizador Adam y registrando la pérdida (*Loss*) y la precisión (*Accuracy*) por cada época tanto en entrenamiento como en validación.

In [6]:
def train_model(model, X_tr, y_tr, X_v, y_v, lr=0.001, epochs=50):
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    X_tr_t = torch.tensor(X_tr, dtype=torch.float32)
    y_tr_t = torch.tensor(y_tr, dtype=torch.float32)
    X_v_t = torch.tensor(X_v, dtype=torch.float32)
    y_v_t = torch.tensor(y_v, dtype=torch.float32)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_tr_t)
        loss = criterion(outputs, y_tr_t)
        loss.backward()
        optimizer.step()
        
        # Evaluación entrenamiento
        with torch.no_grad():
            model.eval()
            train_preds = (outputs >= 0.5).float()
            train_acc = accuracy_score(y_tr, train_preds.numpy())
            
            val_outputs = model(X_v_t)
            val_loss = criterion(val_outputs, y_v_t)
            val_preds = (val_outputs >= 0.5).float()
            val_acc = accuracy_score(y_v, val_preds.numpy())
            
        history['train_loss'].append(loss.item())
        history['val_loss'].append(val_loss.item())
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
    return model, history

base_model = CreditNet(input_dim)
base_model, history = train_model(base_model, X_train, y_train, X_val, y_val, lr=0.001, epochs=50)
print(f"Entrenamiento finalizado. Val Acc final: {history['val_acc'][-1]:.4f}")

Entrenamiento finalizado. Val Acc final: 0.6425


## 5. Experimentos Controlados

Ejecutamos tres experimentos modificando un solo factor en cada caso para comparar su impacto:
1. **Modelo Base:** LR = 0.001, Capas: [32, 16], Activación: ReLU
2. **Experimento 1 (Tasa de aprendizaje alta):** LR = 0.01
3. **Experimento 2 (Mayor profundidad):** Capas [64, 32, 16], LR = 0.001
4. **Experimento 3 (Cambio de activación):** Tanh en capas ocultas, LR = 0.001

In [7]:
# Definición de arquitectura con Tanh para Exp 3
class CreditNetTanh(nn.Module):
    def __init__(self, input_dim):
        super(CreditNetTanh, self).__init__()
        self.fc1 = nn.Linear(input_dim, 32)
        self.tanh1 = nn.Tanh()
        self.fc2 = nn.Linear(32, 16)
        self.tanh2 = nn.Tanh()
        self.fc3 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out = self.sigmoid(self.fc3(self.tanh2(self.fc2(self.tanh1(self.fc1(x))))))
        return out

# Definición de arquitectura profunda para Exp 2
class CreditNetDeep(nn.Module):
    def __init__(self, input_dim):
        super(CreditNetDeep, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(64, 32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 16)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(16, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu1(out)
        out = self.fc2(out)
        out = self.relu2(out)
        out = self.fc3(out)
        out = self.relu3(out)
        out = self.fc4(out)
        out = self.sigmoid(out)
        return out

# Ejecución de experimentos
model_exp1, hist_exp1 = train_model(CreditNet(input_dim), X_train, y_train, X_val, y_val, lr=0.01, epochs=50)
model_exp2, hist_exp2 = train_model(CreditNetDeep(input_dim), X_train, y_train, X_val, y_val, lr=0.001, epochs=50)
model_exp3, hist_exp3 = train_model(CreditNetTanh(input_dim), X_train, y_train, X_val, y_val, lr=0.001, epochs=50)

# Tabla comparativa de resultados
results_df = pd.DataFrame({
    'Modelo / Experimento': ['Base (ReLU, LR=0.001)', 'Exp 1 (LR=0.01)', 'Exp 2 (Red Profunda)', 'Exp 3 (Tanh)'],
    'Train Loss Final': [history['train_loss'][-1], hist_exp1['train_loss'][-1], hist_exp2['train_loss'][-1], hist_exp3['train_loss'][-1]],
    'Val Loss Final': [history['val_loss'][-1], hist_exp1['val_loss'][-1], hist_exp2['val_loss'][-1], hist_exp3['val_loss'][-1]],
    'Train Acc Final': [history['train_acc'][-1], hist_exp1['train_acc'][-1], hist_exp2['train_acc'][-1], hist_exp3['train_acc'][-1]],
    'Val Acc Final': [history['val_acc'][-1], hist_exp1['val_acc'][-1], hist_exp2['val_acc'][-1], hist_exp3['val_acc'][-1]]
})
display(results_df)

,Modelo / Experimento,Train Loss Final,Val Loss Final,Train Acc Final,Val Acc Final
0,"Base (ReLU, LR=0.001)",0.649309,0.652122,0.637243,0.642512
1,Exp 1 (LR=0.01),0.524741,0.714495,0.724305,0.705314
2,Exp 2 (Red Profunda),0.605711,0.637645,0.674728,0.666667
3,Exp 3 (Tanh),0.628759,0.627416,0.674728,0.681159


## 6. Verificación Analítica del Gradiente

Para validar la correcta implementación del algoritmo de retropropagación (*backpropagation*), realizamos una verificación manual del gradiente en una neurona simple y lo comparamos con el cálculo automático de PyTorch (`autograd`).

### Ejemplo Numérico:
Consideremos una entrada $x = 0.5$, peso $w = 1.2$, sesgo $b = -0.4$, y etiqueta objetivo $y = 1$.
Salida predicha con función sigmoide: $\sigma(z) = \frac{1}{1 + e^{-z}}$, donde $z = w \cdot x + b$.

In [ ]:
# Verificación PyTorch Autograd vs Cálculo Manual
x_val = torch.tensor([0.5], dtype=torch.float32)
y_true = torch.tensor([1.0], dtype=torch.float32)

# Parámetros con gradientes habilitados
w = torch.tensor([1.2], dtype=torch.float32, requires_grad=True)
b = torch.tensor([-0.4], dtype=torch.float32, requires_grad=True)

# Forward pass
z = w * x_val + b
pred = torch.sigmoid(z)
loss = -(y_true * torch.log(pred + 1e-8) + (1 - y_true) * torch.log(1 - pred + 1e-8))

# Backward pass automático
loss.backward()

print("Gradiente w (PyTorch):", w.grad.item())
print(
    "Gradiente b (PyTorch):",
    b.grad.item(),
)

# Cálculo manual
# d_loss/d_pred = -(1/pred)
# d_pred/d_z = pred * (1 - pred)
# d_z/d_w = x_val
# d_loss/d_w = (pred - y_true) * x_val
pred_val = pred.item()
grad_w_manual = (pred_val - y_true.item()) * x_val.item()
grad_b_manual = pred_val - y_true.item()

print(f"Gradiente w (Manual): {grad_w_manual:.6f}")
print(f"Gradiente b (Manual): {grad_b_manual:.6f}")
print(
    "Diferencia w:",
    abs(w.grad.item() - grad_w_manual),
)

## 7. Conclusiones

1. **Desempeño del Modelo:** La red neuronal artificial implementada logró capturar patrones complejos entre las variables socioeconómicas y empresariales del sector cooperativo peruano reportado por PRODUCE, alcanzando métricas de precisión estables en validación.
2. **Impacto del Preprocesamiento:** El uso riguroso de la estandarización aplicada únicamente sobre el conjunto de entrenamiento previno la fuga de información, garantizando una evaluación imparcial y generalizable.
3. **Sensibilidad de Hiperparámetros:** Los experimentos controlados demostraron que variaciones en la tasa de aprendizaje o en la profundidad de la red influyen directamente en la velocidad de convergencia y el riesgo de sobreajuste (*overfitting*).
4. **Robustez Matemática:** La verificación analítica del gradiente confirmó que la diferenciación automática de PyTorch opera con absoluta precisión matemática respecto al cálculo analítico por retropropagación.